In [0]:
employee_df = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv("/Volumes/sql_problems/default/my_volume/day02_employees.csv")

display(employee_df)

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, row_number

# define the window specification
windowSpec = Window.partitionBy("department_id") \
    .orderBy(col("salary").desc())

# Apply row_number() and filter for top 3
top_earners_df = (
    employee_df \
    .withColumn("row_num", row_number().over(windowSpec))\
    .filter(col("row_num") <= 3)
    .drop("row_num")
)

top_earners_df.show()

In [0]:

employee_df.createOrReplaceTempView("employee")


In [0]:
%sql
WITH RankedEarners AS (
    SELECT 
        employee_id,
        department_id,
        salary,
        ROW_NUMBER() OVER (
            PARTITION BY department_id 
            ORDER BY salary DESC
        ) as row_num
    FROM employee
)
SELECT 
    department_id,
    employee_id,
    salary
FROM RankedEarners
WHERE row_num <= 3;